In [1]:
import sys
sys.path.append('..')
import cv2
import json
import torch
import torch.nn as nn
import numpy as np
import torch.nn.functional as F
from cying.nn import VisionModel
from PIL import Image
from tqdm import tqdm
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from cityscapesscripts.helpers.labels import labels
from torch_linear_assignment import batch_linear_assignment
from torchinfo import summary

label_to_Id = {label.name: label.trainId for label in labels}

class CityscapesDataset(Dataset):
    def __init__(self, root, split, target_size, num_query):
        self.root = Path(root)
        self.split = split
        self.size = target_size
        self.num_query = num_query

        gt_dir = self.root / 'gtFine' / split
        self.json_files = sorted(gt_dir.glob('*/*_gtFine_polygons.json'))

    def __len__(self):
        return len(self.json_files)

    def __getitem__(self, idx):
        json_path = self.json_files[idx]
        with open(json_path, 'r') as f:
            data = json.load(f)

        img_path = Path(str(json_path).replace('_gtFine_polygons.json', '_leftImg8bit.png').replace('gtFine', 'leftImg8bit'))
        image = np.array(Image.open(img_path).convert('RGB'))  

        h, w = image.shape[:2]
        target_h, target_w = self.size
        scale = min(target_h / h, target_w / w)
        new_h, new_w = int(h * scale), int(w * scale)

        image_resized = torch.tensor(cv2.resize(image, (new_w, new_h), interpolation=cv2.INTER_LINEAR), dtype=torch.float32).permute(2, 0, 1) / 255.0
        image = torch.tile(
            image_resized,
            (
                1,
                (target_h + new_h - 1) // new_h, 
                (target_w + new_w - 1) // new_w   
            ),
        )[:, :target_h, :target_w]
        padding_mask = torch.ones((target_h, target_w), dtype=bool)
        padding_mask[:new_h, :new_w] = False

        masks = torch.zeros((self.num_query, target_h, target_w), dtype=torch.uint8)
        labels = torch.full((self.num_query,), -1, dtype=torch.long)
        bboxes = torch.zeros((self.num_query, 4), dtype=torch.float32)

        valid_idx = 0
        for obj in data['objects']:
            label_str = obj['label']
            label_id = label_to_Id.get(label_str, -1)
            if label_id == -1 or label_id == 255:
                continue

            polygon = np.array(obj['polygon'], dtype=np.int32)
            x_coords = polygon[:, 0]
            y_coords = polygon[:, 1]
            x_min, x_max = x_coords.min(), x_coords.max()
            y_min, y_max = y_coords.min(), y_coords.max()

            polygon_scaled = (polygon * scale).astype(np.int32)
            bbox_scaled = [x_min * scale/target_w, y_min * scale/target_h, x_max * scale/target_w, y_max * scale/target_h]

            mask = np.zeros((target_h, target_w), dtype=np.uint8)
            cv2.fillPoly(mask, [polygon_scaled], 1)

            masks[valid_idx] = torch.as_tensor(mask, dtype=torch.uint8)
            labels[valid_idx] = torch.as_tensor(label_id, dtype=torch.long)
            bboxes[valid_idx] = torch.as_tensor(bbox_scaled, dtype=torch.float32)

            valid_idx += 1

        return image, labels, bboxes, masks, padding_mask

In [2]:
cityscapes_root = "/root/autodl-tmp/cityscapes"
target_size = (256, 512)
batch_size = 8
num_query = 291
num_classes = 19
num_heads = 4
decoder_layers = 3
hidden_width = 256

num_epochs = 50
lr = 1e-3
num_workers = 4

device = torch.device("cuda:0")
test_dataset = CityscapesDataset(
    root = cityscapes_root,
    split = "test",
    target_size = target_size,
    num_query = num_query
)

In [3]:
model = torch.load('./cityscapes.pth', weights_only=False)
model.eval()
summary(model)

Layer (type:depth-idx)                        Param #
VisionModel                                   --
├─OperatorModel2d: 1-1                        --
│    └─Sequential: 2-1                        --
│    │    └─OperatorLayer2d: 3-1              4,212
│    │    └─OperatorLayer2d: 3-2              9,488
├─OperatorModel2d: 1-2                        --
│    └─Sequential: 2-2                        --
│    │    └─OperatorLayer2d: 3-3              18,976
│    │    └─OperatorLayer2d: 3-4              33,856
│    │    └─OperatorLayer2d: 3-5              62,848
├─VisionDecoder: 1-3                          74,496
│    └─ModuleList: 2-3                        --
│    │    └─VisionDecoderLayer: 3-6           198,912
│    │    └─VisionDecoderLayer: 3-7           198,912
│    │    └─VisionDecoderLayer: 3-8           198,912
├─VisionPredictionHeads: 1-4                  --
│    └─Linear: 2-4                            2,580
│    └─Sequential: 2-5                        --
│    │    └─Linear: 3-9 

In [6]:
images, labels, bboxes, masks, padding_mask = test_dataset[0]
images = images.to(device, non_blocking=True)
labels = labels.to(device, non_blocking=True)
masks = masks.to(device, non_blocking=True)
bboxes = bboxes.to(device, non_blocking=True)
padding_mask = padding_mask.to(device, non_blocking=True)

logits, boxes, masks = model(images[None,...], padding_mask[None,...])

OutOfMemoryError: CUDA out of memory. Tried to allocate 84.00 MiB. GPU 0 has a total capacity of 31.47 GiB of which 21.31 MiB is free. Process 18437 has 30.35 GiB memory in use. Including non-PyTorch memory, this process has 1.09 GiB memory in use. Of the allocated memory 783.91 MiB is allocated by PyTorch, and 30.09 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)